# VLM-Anomaly — PatchCore Full MVTec Sweep (Kaggle GPU)

**Standalone implementation — no anomalib, no extra pip installs.**  
Uses only packages pre-installed on Kaggle: `torch`, `torchvision`, `sklearn`, `PIL`.

**GPU:** Set *Settings → Accelerator → GPU T4 x2*  
**Dataset:** Add [ipythonx/mvtec-ad](https://www.kaggle.com/datasets/ipythonx/mvtec-ad) via *Add Data*

## After the run
Download `patchcore_mvtec_results.json` from the **Output** tab, then locally:
```bash
cp ~/Downloads/patchcore_mvtec_results.json  <repo>/results/
```


In [ ]:
# ── Cell 1: Verify environment — no pip install needed ───────────────────────
import torch, torchvision, sklearn, numpy as np
from PIL import Image

print(f'torch      : {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')
print(f'numpy      : {np.__version__}')
print(f'sklearn    : {sklearn.__version__}')
print(f'CUDA       : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU        : {p.name}  ({p.total_memory/1e9:.1f} GB)')
else:
    print('WARNING: No GPU — go to Settings -> Accelerator -> GPU T4 x2')


In [ ]:
# ── Cell 2: Find MVTec dataset ───────────────────────────────────────────────
import os
from pathlib import Path

print('Attached datasets in /kaggle/input:')
for d in sorted(Path('/kaggle/input').iterdir()):
    print(f'  {d}')
print()

MVTEC_ROOT = None
EXPECTED = {'bottle', 'cable', 'capsule', 'carpet', 'grid'}
for root, dirs, _ in os.walk('/kaggle/input'):
    if EXPECTED.issubset(set(dirs)):
        MVTEC_ROOT = Path(root)
        break

assert MVTEC_ROOT, (
    'MVTec not found. Add dataset via Edit -> Add Data -> search "mvtec-ad" (by ipythonx)'
)
categories = sorted(d.name for d in MVTEC_ROOT.iterdir() if d.is_dir())
print(f'MVTec root : {MVTEC_ROOT}')
print(f'Categories : {categories}')


In [ ]:
# ── Cell 3: Configure ────────────────────────────────────────────────────────
from pathlib import Path

DEVICE      = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
IMAGE_SIZE  = 256
BATCH_SIZE  = 32
SUBSAMPLE   = 0.10   # fraction of train patches kept in memory bank (10% ≈ paper default)
RESULTS_DIR = Path('/kaggle/working/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device     : {DEVICE}')
print(f'Image size : {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'Subsample  : {SUBSAMPLE*100:.0f}% of training patches')
print(f'Results    : {RESULTS_DIR}')
print(f'MVTec root : {MVTEC_ROOT}')


In [ ]:
# ── Cell 4: Standalone PatchCore ─────────────────────────────────────────────
# Pure torch + sklearn — no anomalib, works with any numpy version.
import json as _json, time, uuid, warnings
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.models as tvm
import torchvision.transforms.v2 as tvt
from torch.utils.data import DataLoader, Dataset
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, f1_score
from PIL import Image
from pathlib import Path
warnings.filterwarnings('ignore')

MODEL_ID = 'classical/patchcore'

# ── Dataset ───────────────────────────────────────────────────────────────────
class _MVTec(Dataset):
    def __init__(self, root, category, split, transform):
        base = Path(root) / category / split
        self.imgs, self.labels = [], []
        if split == 'train':
            good_dir = base / 'good'
            for p in sorted(good_dir.glob('*')):
                if p.suffix.lower() in ('.png', '.jpg', '.bmp'):
                    self.imgs.append(p); self.labels.append(0)
        else:
            for d in sorted(base.iterdir()):
                if not d.is_dir(): continue
                lbl = 0 if d.name == 'good' else 1
                for p in sorted(d.glob('*')):
                    if p.suffix.lower() in ('.png', '.jpg', '.bmp'):
                        self.imgs.append(p); self.labels.append(lbl)
        self.transform = transform

    def __len__(self): return len(self.imgs)

    def __getitem__(self, i):
        img = Image.open(self.imgs[i]).convert('RGB')
        return self.transform(img), self.labels[i]

# ── Feature extractor: WideResNet50, layers 2 + 3 ────────────────────────────
class _Extractor(torch.nn.Module):
    def __init__(self, device):
        super().__init__()
        backbone = tvm.wide_resnet50_2(
            weights=tvm.Wide_ResNet50_2_Weights.IMAGENET1K_V1
        )
        # layer2 output: stride 8  (512 ch)
        # layer3 output: stride 16 (1024 ch)
        children = list(backbone.children())
        self.stem_l2 = torch.nn.Sequential(*children[:6])   # up to layer2
        self.l3      = children[6]                          # layer3 only
        self.to(device).eval()

    @torch.no_grad()
    def forward(self, x):
        f2 = self.stem_l2(x)
        f3 = self.l3(f2)
        return f2, f3

_TRANSFORM = tvt.Compose([
    tvt.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
    tvt.ToTensor(),
    tvt.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def _extract_patches(extractor, loader, device):
    """Return all patch embeddings as numpy array (N_patches, C)."""
    parts = []
    for imgs, _ in loader:
        f2, f3 = extractor(imgs.to(device))
        f3_up = F.interpolate(f3, size=f2.shape[-2:],
                              mode='bilinear', align_corners=False)
        feats = torch.cat([f2, f3_up], dim=1)  # B, 512+1024, H, W
        B, C, H, W = feats.shape
        parts.append(feats.permute(0, 2, 3, 1).reshape(-1, C).cpu().numpy())
    return np.concatenate(parts, axis=0)

# ── Main evaluator ────────────────────────────────────────────────────────────
def run_patchcore(category: str) -> dict:
    train_ds = _MVTec(MVTEC_ROOT, category, 'train', _TRANSFORM)
    test_ds  = _MVTec(MVTEC_ROOT, category, 'test',  _TRANSFORM)
    kw = dict(batch_size=BATCH_SIZE, num_workers=4, pin_memory=(DEVICE=='cuda'))
    train_ld = DataLoader(train_ds, shuffle=False, **kw)
    test_ld  = DataLoader(test_ds,  shuffle=False, **kw)

    extractor = _Extractor(DEVICE)
    t0 = time.perf_counter()

    # ── Build memory bank ─────────────────────────────────────────────────────
    train_patches = _extract_patches(extractor, train_ld, DEVICE)
    n_keep = max(1, int(len(train_patches) * SUBSAMPLE))
    rng    = np.random.default_rng(42)
    keep   = rng.choice(len(train_patches), n_keep, replace=False)
    memory = train_patches[keep]

    nn = NearestNeighbors(n_neighbors=1, algorithm='ball_tree',
                          metric='euclidean', n_jobs=-1)
    nn.fit(memory)

    # ── Score test images ─────────────────────────────────────────────────────
    scores, labels = [], []
    for imgs, lbls in test_ld:
        f2, f3 = extractor(imgs.to(DEVICE))
        f3_up = F.interpolate(f3, size=f2.shape[-2:],
                              mode='bilinear', align_corners=False)
        feats = torch.cat([f2, f3_up], dim=1)
        B, C, H, W = feats.shape
        feats_np = feats.permute(0, 2, 3, 1).reshape(B, -1, C).cpu().numpy()
        for i in range(B):
            dists, _ = nn.kneighbors(feats_np[i])   # H*W distances
            scores.append(float(dists.max()))         # image score = max patch dist
            labels.append(int(lbls[i]))

    elapsed_ms = (time.perf_counter() - t0) * 1000
    scores_a = np.array(scores, dtype=float)
    labels_a = np.array(labels, dtype=int)

    auroc = float(roc_auc_score(labels_a, scores_a)) if len(set(labels_a)) > 1 else 0.0

    # F1 at optimal threshold (sweep percentiles)
    best_f1 = 0.0
    for pct in np.linspace(0, 100, 200):
        thr = float(np.percentile(scores_a, pct))
        f1  = float(f1_score(labels_a, (scores_a >= thr).astype(int),
                             zero_division=0))
        if f1 > best_f1:
            best_f1 = f1

    result = {
        'model_id': MODEL_ID, 'backend': 'torch_scratch',
        'dataset': 'mvtec', 'category': category,
        'n_images': len(test_ds),
        'auroc': auroc, 'f1': best_f1,
        'precision': None, 'recall': None, 'pro_score': None,
        'mean_latency_ms': elapsed_ms, 'total_cost_usd': 0.0,
    }
    out = RESULTS_DIR / f'{uuid.uuid4().hex[:8]}_mvtec_{category}_patchcore.json'
    out.write_text(_json.dumps([result], indent=2))
    print(f'  {category:12s}  AUROC={auroc:.3f}  F1={best_f1:.3f}  {elapsed_ms/1000:.0f}s')
    return result

def _already_done(cat):
    for f in RESULTS_DIR.glob(f'*_mvtec_{cat}_patchcore.json'):
        try:
            rows = _json.loads(f.read_text())
            if isinstance(rows, list) and rows and rows[0].get('model_id') == MODEL_ID:
                return rows[0]
        except Exception:
            pass
    return None

print('PatchCore ready  (torch + sklearn, no anomalib).')


In [ ]:
# ── Cell 5: Run all 15 categories (idempotent — skips already-done) ──────────
from tqdm.auto import tqdm

all_results = []
for category in tqdm(categories, desc=f'PatchCore [{DEVICE}]'):
    done = _already_done(category)
    if done:
        print(f'  [skip] {category}')
        all_results.append(done)
    else:
        all_results.append(run_patchcore(category))

print(f'\nComplete: {len(all_results)}/15 categories.')


In [ ]:
# ── Cell 6: Save combined output ─────────────────────────────────────────────
import json as _json
from pathlib import Path

OUT = Path('/kaggle/working/patchcore_mvtec_results.json')
OUT.write_text(_json.dumps(all_results, indent=2))
print(f'Output : {OUT}  ({OUT.stat().st_size/1024:.1f} KB)')
print(f'Rows   : {len(all_results)}')
print()
print('1. Output tab (right panel) -> download patchcore_mvtec_results.json')
print('2. cp ~/Downloads/patchcore_mvtec_results.json <repo>/results/')


In [ ]:
# ── Cell 7: Summary table ────────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(all_results)
for col in ['auroc', 'f1', 'mean_latency_ms']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Mean AUROC  : {df.auroc.mean():.4f}')
print(f'Mean F1     : {df.f1.mean():.4f}')
print(f'Avg time/cat: {df.mean_latency_ms.mean()/1000:.0f}s')
print(f'Total cost  : $0.00')
print()
display(df[['category','auroc','f1','mean_latency_ms']]
        .sort_values('auroc', ascending=False).reset_index(drop=True))
